# MA3632 — Workshop 12: Graph Analytics

This workshop accompanies Lecture 12. Parts A and B build up graphs as
mathematical objects: adjacency matrices, degree, walks, and connected
components. Part C computes centrality measures, including eigenvector
centrality, on Zachary's Karate Club. Part D covers PageRank. Part E develops
the graph Laplacian and applies spectral clustering to a k-nearest-neighbour
graph built from Digits, the module's primary dataset throughout the course.

Work through all parts in order. Take-home exercises are at the end.

---

## Part A — Graphs, Adjacency Matrices, and Connected Components

We represent a graph by its adjacency matrix, verify the handshake lemma
(Proposition 1.5), and identify connected components directly, using the
two-component graph from Lecture 12, Exercise 1: vertices $\{1,2,3,4,5\}$
with edges $\{1,2\}, \{1,3\}, \{2,3\}, \{4,5\}$.

### A1. Imports and data

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_digits
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score
from scipy.sparse.csgraph import laplacian
from scipy.linalg import eigh

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

digits = load_digits()
X_dig, y_dig = digits.data, digits.target
print(f"Digits — {X_dig.shape[0]} samples, {X_dig.shape[1]} features, {len(np.unique(y_dig))} classes")

### A2. The handshake lemma and connected components

In [ ]:
G_a = nx.Graph()
G_a.add_nodes_from([1, 2, 3, 4, 5])
G_a.add_edges_from([(1, 2), (1, 3), (2, 3), (4, 5)])

A_a = nx.adjacency_matrix(G_a, nodelist=[1,2,3,4,5]).toarray()
print("Adjacency matrix (vertex order 1..5):")
print(A_a)

degrees = dict(G_a.degree())
print(f"\nDegrees: {degrees}")
print(f"Sum of degrees: {sum(degrees.values())}")
print(f"2 * |E|:         {2 * G_a.number_of_edges()}")
print("Handshake lemma holds:", sum(degrees.values()) == 2 * G_a.number_of_edges())

components = list(nx.connected_components(G_a))
print(f"\nConnected components: {components}")

fig, ax = plt.subplots(figsize=(4, 3))
pos = nx.spring_layout(G_a, seed=1)
nx.draw(G_a, pos, ax=ax, with_labels=True, node_color='steelblue',
        node_size=500, font_color='white', font_weight='bold')
plt.title('Two-component graph (Lecture 12, Exercise 1)')
plt.savefig('/tmp/a_graph.png', dpi=110, bbox_inches='tight')
plt.close()
print("Saved a_graph.png")

### A3. Adjacency matrices as heatmaps, and a weighted example

Adjacency matrices for graphs with more than a handful of vertices are
easier to read as heatmaps than as printed arrays. We also extend the
definition from Lecture 12, Section 1 to the weighted case: instead of a
0/1 entry, $A_{ij}$ carries the edge weight $w_{ij}$, and the handshake
lemma generalises to $\sum_i \deg_w(i) = 2\sum_{\{i,j\} \in E} w_{ij}$,
where $\deg_w(i) = \sum_j A_{ij}$ is the *weighted* degree.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(A_a, cmap='Blues')
axes[0].set_xticks(range(5)); axes[0].set_xticklabels([1,2,3,4,5])
axes[0].set_yticks(range(5)); axes[0].set_yticklabels([1,2,3,4,5])
axes[0].set_title('Part A graph (unweighted)')
for (i,j), v in np.ndenumerate(A_a):
    axes[0].text(j, i, int(v), ha='center', va='center',
                 color='white' if v > 0 else 'black')

# A weighted version of the same graph: attach distances to the two
# triangle edges and the bridge edge, e.g. as if vertices were locations
# and weights were travel times in minutes.
G_a_w = nx.Graph()
G_a_w.add_weighted_edges_from([(1,2,3.0), (1,3,1.0), (2,3,2.0), (4,5,4.0)])
A_a_w = nx.adjacency_matrix(G_a_w, nodelist=[1,2,3,4,5], weight='weight').toarray()

im = axes[1].imshow(A_a_w, cmap='Blues')
axes[1].set_xticks(range(5)); axes[1].set_xticklabels([1,2,3,4,5])
axes[1].set_yticks(range(5)); axes[1].set_yticklabels([1,2,3,4,5])
axes[1].set_title('Same graph, weighted')
for (i,j), v in np.ndenumerate(A_a_w):
    axes[1].text(j, i, f'{v:.0f}' if v > 0 else '0', ha='center', va='center',
                 color='white' if v > 2 else 'black')
plt.tight_layout()
plt.savefig('/tmp/a3_heatmaps.png', dpi=110, bbox_inches='tight')
plt.close()
print('Saved a3_heatmaps.png')

In [ ]:
weighted_degrees = dict(G_a_w.degree(weight='weight'))
total_weight = sum(d['weight'] for _,_,d in G_a_w.edges(data=True))
print(f"Weighted degrees: {weighted_degrees}")
print(f"Sum of weighted degrees: {sum(weighted_degrees.values())}")
print(f"2 * (total edge weight):  {2 * total_weight}")
print("Weighted handshake lemma holds:",
      np.isclose(sum(weighted_degrees.values()), 2 * total_weight))

**Exercise A.** Add a single edge between the two components above (e.g.
$\{3,4\}$) and recompute `nx.connected_components`. How many components does
the graph now have, and what does Theorem 5.3 (the eigenvalue-0 multiplicity
result) predict for the dimension of the Laplacian's null space as a result?
Verify your prediction by computing the Laplacian's eigenvalues directly.

## Part B — Walks and Powers of the Adjacency Matrix

We verify Proposition 2.2: $(A^k)_{ij}$ equals the number of walks of length
$k$ from vertex $i$ to vertex $j$. We use the 4-vertex example from Lecture
12, Section 1: edges $\{1,2\}, \{1,3\}, \{2,3\}, \{3,4\}$.

In [ ]:
G_b = nx.Graph()
G_b.add_nodes_from([1, 2, 3, 4])
G_b.add_edges_from([(1, 2), (1, 3), (2, 3), (3, 4)])
nodelist_b = [1, 2, 3, 4]
A_b = nx.adjacency_matrix(G_b, nodelist=nodelist_b).toarray()

print("A =")
print(A_b)
print("\nA^2 (walks of length 2) =")
print(A_b @ A_b)
print("\nA^3 (walks of length 3) =")
print(np.linalg.matrix_power(A_b, 3))

# Manual sanity check: enumerate walks of length 2 from vertex 1 to vertex 3
# by hand, and compare to (A^2)_{1,3}
walks_1_to_3_len2 = []
for mid in G_b.neighbors(1):
    if G_b.has_edge(mid, 3):
        walks_1_to_3_len2.append((1, mid, 3))
print(f"\nWalks of length 2 from vertex 1 to vertex 3 (enumerated): {walks_1_to_3_len2}")
idx1, idx3 = nodelist_b.index(1), nodelist_b.index(3)
print(f"(A^2)_{{1,3}} from matrix power: {(A_b @ A_b)[idx1, idx3]}")
print("Match:", len(walks_1_to_3_len2) == (A_b @ A_b)[idx1, idx3])

### B2. Counting triangles from $A^3$

Proposition 2.2 also gives a direct way to count triangles: a *closed*
walk of length 3 from a vertex back to itself corresponds to a triangle
through that vertex traversed in one of two directions, so $(A^3)_{ii}$
counts each triangle through vertex $i$ twice. Summing over all vertices
and dividing by $2 \cdot 3 = 6$ (two directions, three vertices per
triangle) gives the total number of triangles in the graph:
$$\#\text{triangles} = \frac{1}{6}\operatorname{tr}(A^3).$$

In [ ]:
A3_power = np.linalg.matrix_power(A_b, 3)
triangle_count_formula = np.trace(A3_power) / 6
triangle_count_direct = sum(nx.triangles(G_b).values()) / 3

print(f"trace(A^3) = {np.trace(A3_power)}")
print(f"Triangles from trace(A^3)/6:        {triangle_count_formula}")
print(f"Triangles from nx.triangles/3:       {triangle_count_direct}")
print("Match:", np.isclose(triangle_count_formula, triangle_count_direct))

# Apply the same check to the Karate Club graph as a larger example.
# Careful: nx.karate_club_graph() edges carry a 'weight' attribute (tie
# strength), and nx.adjacency_matrix uses edge weights by default, so we
# must pass weight=None explicitly to get the binary adjacency matrix the
# formula assumes.
G_karate_b2 = nx.karate_club_graph()
A_karate = nx.adjacency_matrix(G_karate_b2, weight=None).toarray()
karate_triangles_formula = np.trace(np.linalg.matrix_power(A_karate, 3)) / 6
karate_triangles_direct = sum(nx.triangles(G_karate_b2).values()) / 3
print(f"\nKarate Club triangles from trace(A^3)/6: {karate_triangles_formula:.0f}")
print(f"Karate Club triangles from nx.triangles:  {karate_triangles_direct:.0f}")

**Exercise B.** Using the printed $A^3$ above, verify by hand enumeration
that $(A^3)_{2,4}$ is correct: list every walk of length 3 from vertex 2 to
vertex 4 in this graph, and check that the count matches the matrix entry.

## Part C — Centrality Measures

We first verify the star-graph eigenvector centrality worked example from
Lecture 12 (Example 3.5), then compute all four centrality measures on
Zachary's Karate Club graph — a classic, small (34-vertex) social network
built into `networkx` and a standard teaching example in network science.

### C1. Verifying the star graph example

In [ ]:
# Hand-derived: centre eigenvector centrality should be 1/sqrt(2) approx 0.7071,
# each leaf 1/sqrt(6) approx 0.4082
G_star = nx.star_graph(3)   # centre 0, leaves 1, 2, 3
star_centrality = nx.eigenvector_centrality(G_star)
print("Eigenvector centrality (networkx):", star_centrality)
print(f"Expected: centre = {1/np.sqrt(2):.4f}, leaf = {1/np.sqrt(6):.4f}")

### C2. Centrality on Zachary's Karate Club

In [ ]:
G_c = nx.karate_club_graph()
print(f"Karate club graph: {G_c.number_of_nodes()} vertices, {G_c.number_of_edges()} edges")

degree_c      = nx.degree_centrality(G_c)
closeness_c   = nx.closeness_centrality(G_c)
betweenness_c = nx.betweenness_centrality(G_c)
eigenvector_c = nx.eigenvector_centrality(G_c)

def top5(centrality_dict, name):
    ranked = sorted(centrality_dict.items(), key=lambda kv: kv[1], reverse=True)[:5]
    print(f"\nTop 5 by {name}:")
    for v, score in ranked:
        print(f"  vertex {v:>2}: {score:.4f}")

top5(degree_c, "degree centrality")
top5(closeness_c, "closeness centrality")
top5(betweenness_c, "betweenness centrality")
top5(eigenvector_c, "eigenvector centrality")

In [ ]:
# Visualise the graph with node size proportional to eigenvector centrality
fig, ax = plt.subplots(figsize=(6, 5))
pos_c = nx.spring_layout(G_c, seed=7)
sizes = [4000 * eigenvector_c[v] for v in G_c.nodes()]
nx.draw(G_c, pos_c, ax=ax, with_labels=True, node_color='steelblue',
        node_size=sizes, font_size=8, font_color='white')
plt.title('Zachary Karate Club: node size = eigenvector centrality')
plt.savefig('/tmp/c_karate_centrality.png', dpi=110, bbox_inches='tight')
plt.close()
print("Saved c_karate_centrality.png")

In [ ]:
# Compare betweenness centrality against eigenvector centrality directly:
# a vertex can score highly on one without scoring highly on the other,
# since betweenness rewards bridging position while eigenvector centrality
# rewards proximity to other well-connected vertices.
vertices_c = list(G_c.nodes())
between_vals = [betweenness_c[v] for v in vertices_c]
eigen_vals   = [eigenvector_c[v] for v in vertices_c]

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(eigen_vals, between_vals, color='steelblue')
for v, x, y in zip(vertices_c, eigen_vals, between_vals):
    if y > 0.1 or x > 0.3:
        ax.annotate(str(v), (x, y), textcoords='offset points', xytext=(4,4), fontsize=8)
ax.set_xlabel('Eigenvector centrality')
ax.set_ylabel('Betweenness centrality')
ax.set_title('Betweenness vs eigenvector centrality (Karate Club)')
plt.tight_layout()
plt.savefig('/tmp/c_centrality_scatter.png', dpi=110, bbox_inches='tight')
plt.close()
print("Saved c_centrality_scatter.png")

corr = np.corrcoef(eigen_vals, between_vals)[0,1]
print(f"Correlation between eigenvector and betweenness centrality: {corr:.3f}")

### C3. A combined centrality table, and robustness to removing a vertex

It is useful to see all four centrality measures side by side rather than
as separate top-5 lists. We build a single table and also check how
sensitive the rankings are to a small perturbation: removing the single
highest-degree vertex (the instructor, vertex 0) and recomputing.

In [ ]:
import pandas as pd

centrality_table = pd.DataFrame({
    'degree': degree_c,
    'closeness': closeness_c,
    'betweenness': betweenness_c,
    'eigenvector': eigenvector_c,
}).sort_values('eigenvector', ascending=False)
print(centrality_table.head(8).round(4))

print(f"\nSpearman rank correlations between the four measures:")
print(centrality_table.corr(method='spearman').round(3))

In [ ]:
# Remove the highest-degree vertex (vertex 0, the instructor) and recompute
G_c_perturbed = G_c.copy()
G_c_perturbed.remove_node(0)

eigenvector_perturbed = nx.eigenvector_centrality(G_c_perturbed)

# Compare rankings for the vertices that survive, using Spearman correlation
common_vertices = [v for v in G_c.nodes() if v != 0]
before = [eigenvector_c[v] for v in common_vertices]
after = [eigenvector_perturbed[v] for v in common_vertices]
from scipy.stats import spearmanr
rho, _ = spearmanr(before, after)
print(f"Vertices remaining after removing vertex 0: {G_c_perturbed.number_of_nodes()}")
print(f"Is the perturbed graph still connected? {nx.is_connected(G_c_perturbed)}")
print(f"Spearman rank correlation of eigenvector centrality, before vs after removal: {rho:.3f}")
print("A high correlation indicates the ranking is dominated by graph structure")
print("away from vertex 0, rather than by vertex 0 itself.")

**Exercise C.** The Karate Club graph famously splits into two factions after
a real-world conflict between the instructor (vertex 0) and the administrator
(vertex 33). Using the scatter plot above, identify any vertex that ranks
noticeably higher on betweenness than its eigenvector centrality would
suggest (or vice versa), and explain, in terms of the two definitions, what
structural role such a vertex is likely playing in the network.

## Part D — PageRank

We reproduce the hand-worked PageRank example from Lecture 12 (Example 4.3):
the directed graph $A \to B$, $A \to C$, $B \to C$, $C \to A$, with damping
factor $\alpha = 0.5$ (matching the lecture) and $\alpha = 0.85$ (matching
Exercise 2).

In [ ]:
G_d = nx.DiGraph()
G_d.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'A')])

pagerank_05  = nx.pagerank(G_d, alpha=0.5)
pagerank_085 = nx.pagerank(G_d, alpha=0.85)

print("PageRank, alpha=0.5:  ", pagerank_05)
print("Expected (exact):      A=14/39=%.4f  B=10/39=%.4f  C=15/39=%.4f" % (14/39, 10/39, 15/39))
print()
print("PageRank, alpha=0.85: ", pagerank_085)

In [ ]:
# PageRank on the (undirected) Karate Club graph, compared to eigenvector centrality
pagerank_karate = nx.pagerank(G_c, alpha=0.85)

top5(pagerank_karate, "PageRank (alpha=0.85)")

# Rank correlation between PageRank and eigenvector centrality orderings
pr_rank  = np.argsort(np.argsort([-pagerank_karate[v] for v in vertices_c]))
ev_rank  = np.argsort(np.argsort([-eigenvector_c[v] for v in vertices_c]))
rank_agreement = np.mean(np.abs(pr_rank - ev_rank) <= 2)
print(f"\nFraction of vertices whose PageRank and eigenvector-centrality ranks "
      f"differ by at most 2 positions: {rank_agreement:.2f}")

### D2. PageRank by power iteration

Remark 4.2 describes PageRank as the stationary distribution of a random
surfer. We compute that stationary distribution directly by power
iteration on the Google matrix $M = \alpha P + (1-\alpha) \frac{1}{n} J$,
where $P$ is the (column-)normalised transition matrix and $J$ is the
all-ones matrix, and track how quickly the iterates converge to the
values `nx.pagerank` reports.

In [ ]:
nodelist_d = ['A', 'B', 'C']
A_d = nx.adjacency_matrix(G_d, nodelist=nodelist_d).toarray().astype(float)
n_d = len(nodelist_d)

out_degree = A_d.sum(axis=1)
P = (A_d.T / out_degree).T  # row i normalised by vertex i's out-degree
alpha_pi = 0.85
M = alpha_pi * P.T + (1 - alpha_pi) * np.ones((n_d, n_d)) / n_d

x = np.ones(n_d) / n_d
history = [x.copy()]
for iteration in range(30):
    x = M @ x
    x = x / x.sum()
    history.append(x.copy())
history = np.array(history)

target = np.array([pagerank_085[v] for v in nodelist_d])
print(f"Power iteration, final estimate: {dict(zip(nodelist_d, x.round(4)))}")
print(f"nx.pagerank (alpha=0.85):         {dict(zip(nodelist_d, target.round(4)))}")
print(f"Max absolute difference after 30 iterations: {np.max(np.abs(x - target)):.2e}")

fig, ax = plt.subplots(figsize=(5.5, 4))
for j, v in enumerate(nodelist_d):
    ax.plot(history[:, j], marker='o', markersize=3, label=f'vertex {v}')
    ax.axhline(target[j], color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Iteration')
ax.set_ylabel('PageRank estimate')
ax.set_title('Power iteration convergence (dotted lines: nx.pagerank targets)')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/d2_power_iteration.png', dpi=110, bbox_inches='tight')
plt.close()
print('Saved d2_power_iteration.png')

**Exercise D.** Recompute PageRank on the directed graph above with a very
small damping factor, e.g. $\alpha = 0.05$. What does the resulting ranking
look like, and why does the random-surfer interpretation from Lecture 12,
Remark 4.2 predict this?

## Part E — The Graph Laplacian and Spectral Clustering

This is the capstone of the module: we verify the eigenvalue-0 multiplicity
theorem directly, then turn Digits into a graph, cluster it using the
Laplacian eigendecomposition (Definition 5.5), and compare the result
directly against plain k-means on the raw pixel features — the same
comparison computed directly below, now with a graph-based
alternative in the mix.

### E1. Null space dimension equals the number of connected components

In [ ]:
# Disjoint-triangles graph from Lecture 12, Exercise 3: two triangles
# {1,2,3} and {4,5,6} with no edges between them.
G_e1 = nx.Graph()
G_e1.add_edges_from([(1,2),(2,3),(1,3), (4,5),(5,6),(4,6)])

L_e1 = nx.laplacian_matrix(G_e1, nodelist=[1,2,3,4,5,6]).toarray().astype(float)
print("Laplacian:")
print(L_e1)

eigvals_e1 = np.linalg.eigvalsh(L_e1)
print(f"\nEigenvalues: {eigvals_e1}")
n_zero = np.sum(np.abs(eigvals_e1) < 1e-9)
print(f"Number of (near-)zero eigenvalues: {n_zero}")
print(f"Number of connected components:   {nx.number_connected_components(G_e1)}")
print("Theorem 5.3 confirmed:", n_zero == nx.number_connected_components(G_e1))

# Verify the two indicator vectors from Lecture 12's solution both lie in the null space
x1 = np.array([1,1,1,0,0,0], dtype=float)
x2 = np.array([0,0,0,1,1,1], dtype=float)
print(f"\nL @ x1 = {L_e1 @ x1}  (should be all zeros)")
print(f"L @ x2 = {L_e1 @ x2}  (should be all zeros)")

### E2. Spectral clustering on a k-nearest-neighbour graph built from Digits

In [ ]:
# Subsample to keep the dense eigendecomposition fast: 60 examples per digit
idx = []
for c in range(10):
    class_idx = np.where(y_dig == c)[0]
    idx.extend(rng.choice(class_idx, size=60, replace=False))
idx = np.array(idx)
X_sub, y_sub = X_dig[idx], y_dig[idx]
print(f"Subsample: {X_sub.shape[0]} observations, {len(np.unique(y_sub))} digit classes")

In [ ]:
# Build a k-nearest-neighbour similarity graph and symmetrise it
k_neighbours = 10
knn_graph = kneighbors_graph(X_sub, n_neighbors=k_neighbours, mode='connectivity', include_self=False)
knn_graph = 0.5 * (knn_graph + knn_graph.T)
knn_graph.data[:] = 1.0   # binary adjacency after symmetrising

# Diagnostic: degree distribution of the constructed graph. A k-NN graph is
# not exactly k-regular after symmetrising (a point can be one of another
# point's k nearest neighbours without the reverse holding), so some spread
# around k_neighbours is expected.
degrees_knn = np.asarray(knn_graph.sum(axis=1)).flatten()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(degrees_knn, bins=range(int(degrees_knn.min()), int(degrees_knn.max())+2),
        color='steelblue', edgecolor='white')
ax.axvline(k_neighbours, color='crimson', linestyle='--', label=f'k = {k_neighbours}')
ax.set_xlabel('Vertex degree')
ax.set_ylabel('Count')
ax.set_title('Degree distribution of the Digits k-NN graph')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/e2_degree_dist.png', dpi=110, bbox_inches='tight')
plt.close()
print(f"Mean degree: {degrees_knn.mean():.2f}, min: {degrees_knn.min():.0f}, max: {degrees_knn.max():.0f}")
print("Saved e2_degree_dist.png")

In [ ]:
# Graph Laplacian (normalised, following Remark 5.6) and its smallest eigenvectors
L_knn = laplacian(knn_graph, normed=True)
eigvals_knn, eigvecs_knn = eigh(L_knn.toarray())

n_clusters = 10
U = eigvecs_knn[:, :n_clusters]   # embedding: one row per vertex, n_clusters columns

labels_spectral = KMeans(n_clusters=n_clusters, n_init=10, random_state=0).fit_predict(U)
ari_spectral = adjusted_rand_score(y_sub, labels_spectral)

# Compare to plain k-means on the raw pixel features, computed here as a baseline
labels_raw_kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=0).fit_predict(X_sub)
ari_raw_kmeans = adjusted_rand_score(y_sub, labels_raw_kmeans)

print(f"Spectral clustering (Laplacian eigenvectors + k-means):  ARI = {ari_spectral:.4f}")
print(f"Plain k-means on raw pixel features (baseline):   ARI = {ari_raw_kmeans:.4f}")
print(f"Improvement from using graph structure: {ari_spectral - ari_raw_kmeans:+.4f}")

In [ ]:
# Cross-check against sklearn's built-in SpectralClustering estimator
sklearn_spectral = SpectralClustering(
    n_clusters=n_clusters, affinity='nearest_neighbors', n_neighbors=k_neighbours,
    assign_labels='kmeans', random_state=0
)
labels_sklearn = sklearn_spectral.fit_predict(X_sub)
ari_sklearn = adjusted_rand_score(y_sub, labels_sklearn)
agreement = adjusted_rand_score(labels_spectral, labels_sklearn)

print(f"sklearn SpectralClustering:                ARI vs true labels = {ari_sklearn:.4f}")
print(f"Agreement between our manual implementation and sklearn's:      {agreement:.4f}")

In [ ]:
# Visualise the Laplacian eigenvalue spectrum: the first n_clusters eigenvalues
# should be noticeably smaller than the rest if the ten digit classes form
# reasonably well-separated groups in the k-NN graph
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(eigvals_knn[:30], 'o-', color='steelblue')
ax.axvline(n_clusters - 0.5, color='crimson', linestyle='--',
           label=f'{n_clusters} clusters used')
ax.set_xlabel('Eigenvalue index (smallest to largest)')
ax.set_ylabel('Eigenvalue')
ax.set_title('Laplacian spectrum of the Digits k-NN graph')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/e2_spectrum.png', dpi=110, bbox_inches='tight')
plt.close()
print("Saved e2_spectrum.png")

### E3. Sensitivity to the number of neighbours $k$, and a 2D embedding

The k-NN graph built in E2 depends on the choice of $k$. We sweep over
several values and track how the spectral clustering ARI responds, then
visualise the first two non-trivial Laplacian eigenvectors directly as a
2D embedding of the digits, coloured by true class, to see the cluster
structure that the k-means step in E2 is operating on.

In [ ]:
k_values = [5, 10, 15, 20, 30, 50]
ari_by_k = []
for k_try in k_values:
    knn_try = kneighbors_graph(X_sub, n_neighbors=k_try, mode='connectivity', include_self=False)
    knn_try = 0.5 * (knn_try + knn_try.T)
    knn_try.data[:] = 1.0
    L_try = laplacian(knn_try, normed=True)
    eigvals_try, eigvecs_try = eigh(L_try.toarray())
    U_try = eigvecs_try[:, :n_clusters]
    labels_try = KMeans(n_clusters=n_clusters, n_init=10, random_state=0).fit_predict(U_try)
    ari_by_k.append(adjusted_rand_score(y_sub, labels_try))

for k_try, ari_try in zip(k_values, ari_by_k):
    print(f"k = {k_try:>3}:  ARI = {ari_try:.4f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(k_values, ari_by_k, 'o-', color='steelblue')
ax.axvline(k_neighbours, color='crimson', linestyle='--', label=f'k = {k_neighbours} used in E2')
ax.set_xlabel('Number of neighbours k')
ax.set_ylabel('Adjusted Rand Index')
ax.set_title('Spectral clustering ARI as a function of k')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/e3_k_sensitivity.png', dpi=110, bbox_inches='tight')
plt.close()
print('Saved e3_k_sensitivity.png')

In [ ]:
# The first eigenvector (eigenvalue 0) is constant and carries no
# clustering information; the next two smallest give a 2D embedding.
embedding_2d = eigvecs_knn[:, 1:3]

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=y_sub,
                      cmap='tab10', s=15)
legend1 = ax.legend(*scatter.legend_elements(), title='Digit', loc='upper right',
                     fontsize=7, ncol=2)
ax.add_artist(legend1)
ax.set_xlabel('Laplacian eigenvector 2')
ax.set_ylabel('Laplacian eigenvector 3')
ax.set_title('2D spectral embedding of the Digits k-NN graph (k=10), coloured by true label')
plt.tight_layout()
plt.savefig('/tmp/e3_embedding.png', dpi=110, bbox_inches='tight')
plt.close()
print('Saved e3_embedding.png')
print("Digit classes that already separate cleanly in just these two")
print("coordinates are the ones k-means in E2 recovers most reliably.")

**Exercise E.** Repeat the spectral clustering comparison above using the
*unnormalised* Laplacian ($L = D - A$) instead of the normalised version
used in `E2` above. Does the ARI change much? Given Remark 5.6, explain what
property of the k-NN graph would make normalisation matter more or less.

---

## Take-home exercises

**Exercise 1 — Fewer clusters, cleaner separation.**
Repeat the spectral clustering comparison from Part E2 using only three digit
classes of your choice (e.g. 0, 1, and 8, which are visually similar and
often confused). Does the gap between spectral clustering's ARI and plain
k-means's ARI widen or narrow compared to the full ten-class comparison
above? Try at least two different values of `k_neighbours` (e.g. 5 and 20)
and report how sensitive the result is to this choice.

**Exercise 2 — Walks on a path graph.**
Build the path graph on 5 vertices, $1-2-3-4-5$ (edges $\{1,2\}, \{2,3\},
\{3,4\}, \{4,5\}$, no other edges). Verify the handshake lemma (Part A) and
compute $A^2$ and $A^4$ to check Proposition 2.2 (Part B) by enumerating
walks of length 2 and length 4 between vertex 1 and vertex 5 by hand,
comparing your count to the matrix entry.

**Exercise 3 — Directed PageRank on the Karate Club.**
Construct a directed version of the Karate Club graph by replacing every
undirected edge with two directed edges, one in each direction, and compute
its PageRank with $\alpha = 0.85$. Compare the resulting ranking to the
undirected eigenvector centrality ranking computed in Part C. Do the same
vertices come out on top? Explain what you would expect in general when an
undirected graph is converted to a directed graph in this symmetric way.